## Phase 2.1 Customer Segmentation Churn Analysis

**Objective**: Analyze churn rate of customer segments across multiple dimensions and identify high-churn segments

**Tasks:** 
1. Analyze churn rate by acquisition channel (acquisition_channel)
2. Analyze churn rate by product tier (product_tier)
3. Analyze churn rate by sales segment (sales_segment)
4. Analyze churn rate by region (region)
5. Analyze churn rate by company size (company_size_bucket)
6. Analyze churn rate by onboarding score (initial_onboarding_score)
7. Identify segment with highest churn rate (apply 80/20 principle)

## Step 1. Import requirements

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

## Step 2. Load data

In [2]:
df = pd.read_csv("dataset1_cleaned.csv")
df.head()

,customer_id,company_name,country,region,is_eu,industry,company_size_bucket,annual_contract_value,product_tier,sales_segment,acquisition_channel,contract_start_date,contract_end_date,renewed_flag,discount_pct,initial_onboarding_score,is_churned
0,C100000,Company_0,Canada,North America,0,Healthcare,51-200,12999.25,Growth,MidMarket,Partner,2023-05-02,NaN,1,0.03,7.0,0
1,C100001,Company_1,UK,Europe,1,Wholesale,10-Jan,1799.72,Starter,SMB_Field,Inbound,2023-02-18,NaN,1,0.15,4.9,0
2,C100002,Company_2,US,North America,0,Manufacturing,10-Jan,1770.83,Starter,SMB_Field,Inbound,2023-11-12,2024-05-13,0,0.37,4.8,1
3,C100003,Company_3,France,Europe,1,Unknown,10-Jan,1790.30,Starter,SMB_Field,Outbound,2023-08-06,NaN,1,0.10,8.0,0
4,C100004,Company_4,Netherlands,Europe,1,Professional Services,10-Jan,1552.00,Starter,SMB_Inside,Inbound,2023-09-09,2024-03-10,0,0.11,3.8,1


In [3]:
# Some values in company_size_bucekt column are in date format. Change them to numerical range.
df['company_size_bucket'].unique()

array(['51-200', '10-Jan', '201-1000', 'Nov-50', '1000+'], dtype=object)

In [4]:
df['company_size_bucket'] = (df['company_size_bucket'].replace({'10-Jan': '1-10','Nov-50': '11-50'}))
df['company_size_bucket'].unique()

array(['51-200', '1-10', '201-1000', '11-50', '1000+'], dtype=object)

## Step 3. Multi-dimensional churn analysis (with visualization)
Note:
- With regards to the time horizon, we focus on Q3, when the "crisis" occurred
- While churn can be defined in several different ways in practice, we consider only those who terminated contracts before natural expiration of 1-year term as churned; this could be a deliberate analytical choice to isolate true risk behavior (customer dissatisfaction) from normal lifecycle completion.

In [5]:
# churn rate by acquisition channels 
channels = df['acquisition_channel'].unique()

churn_rates = [] 
for channel in channels: 
    channel = df[df['acquisition_channel']==channel] 
    churn = channel[channel['is_churned']==1] 
    churn_rate = 100*(len(churn)/len(channel)) 
    churn_rates.append(round(churn_rate, 1)) 

plot_df = pd.DataFrame({ 'Acquisition Channel': channels, 'Churn Rate (%)': churn_rates }) 

fig = px.bar( plot_df, x='Acquisition Channel', y='Churn Rate (%)', title='Churn Rate by Acquisition Channel', text='Churn Rate (%)' ) 
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside') 
fig.update_layout( yaxis_title='Churn Rate (%)', xaxis_title='Acquisition Channel', width=800, height=500 ) 
fig.show()

In [6]:
# churn rate by product tiers
tiers = df['product_tier'].unique()
# change the order of disply to match intuition
order = ['Starter', 'Growth', 'Enterprise']
tiers = sorted(tiers, key=lambda x: order.index(x))

churn_rates = []
for tier in tiers:
    tier = df[df['product_tier']==tier]
    churn = tier[tier['is_churned'] == 1]
    churn_rate = 100*(len(churn)/len(tier))
    churn_rates.append(round(churn_rate, 1))

plot_df = pd.DataFrame({'Product Tier': tiers,'Churn Rate (%)': churn_rates})

fig = px.bar(plot_df, x='Product Tier', y='Churn Rate (%)', title='Churn Rate by Product Tier', text='Churn Rate (%)')
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_title='Churn Rate (%)', xaxis_title='Product Tier', width=800, height=500)
fig.show()

In [7]:
# churn rate by sales_segment
# use pandas aggregation for conciseness
plot_df = (
    df.groupby('sales_segment')['is_churned']
      .mean()
      .mul(100)
      .round(1)
      .reset_index(name='Churn Rate (%)')
)

fig = px.bar(plot_df, x='sales_segment', y='Churn Rate (%)', title='Churn Rate by Sales Segment', text='Churn Rate (%)')
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(xaxis_title='Sales Segment', yaxis_title='Churn Rate (%)', width=800, height=500)
fig.show()

In [8]:
# churn rate by region
plot_df = (
    df.groupby('region')['is_churned']
      .mean()
      .mul(100)
      .round(1)
      .reset_index(name='Churn Rate (%)')
)

fig = px.bar(plot_df, x='region', y='Churn Rate (%)', title='Churn Rate by Region', text='Churn Rate (%)')
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(xaxis_title='Region', yaxis_title='Churn Rate (%)', width=600, height=400)
fig.show()

In [9]:
# churn rate by company size
order = ['1-10', '11-50', '51-200', '201-1000', '1000+']

plot_df = (
    df.groupby('company_size_bucket')['is_churned']
      .mean()
      .mul(100)
      .round(1)
      .reindex(order)   # enforce size order
      .reset_index(name='Churn Rate (%)')
)

fig = px.bar(plot_df, x='company_size_bucket', y='Churn Rate (%)', title='Churn Rate by Company Size', text='Churn Rate (%)')
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(xaxis_title='Company Size', yaxis_title='Churn Rate (%)', width=650, height=500)
fig.show()

In [ ]:
# Correlation & Significance (Logistic regression)
# if coefficient is negative → higher onboarding score means lower churn
# # if p-value < 0.001 → statistically significant
import statsmodels.api as sm

X = sm.add_constant(df['initial_onboarding_score'])
y = df['is_churned']

model = sm.Logit(y, X).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.365088
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:             is_churned   No. Observations:                 3000
Model:                          Logit   Df Residuals:                     2998
Method:                           MLE   Df Model:                            1
Date:                Sun, 08 Feb 2026   Pseudo R-squ.:                 0.01213
Time:                        17:41:24   Log-Likelihood:                -1095.3
converged:                       True   LL-Null:                       -1108.7
Covariance Type:            nonrobust   LLR p-value:                 2.133e-07
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                       -0.8396      0.222     -3.788      0.000      -1.274

In [ ]:
# Create bins to segment customers
scores = df['initial_onboarding_score'].unique()
print(scores.min(), scores.max())

1.1 10.0


In [11]:
def score_bucket(score):
    if score < 4:
        return 'Low'
    elif score < 7:
        return 'Average'
    else:
        return 'High'

df['onboarding_score_bucket'] = df['initial_onboarding_score'].apply(score_bucket)

In [12]:
order = ['Low', 'Average', 'High']

plot_df = (
    df.groupby('onboarding_score_bucket')['is_churned']
      .mean()
      .mul(100)
      .round(1)
      .reindex(order)   # enforce bucket order
      .reset_index(name='Churn Rate (%)')
)

fig = px.bar(plot_df, x='onboarding_score_bucket', y='Churn Rate (%)', title='Churn Rate by Onboarding Score', text='Churn Rate (%)')
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(xaxis_title='Onboarding Score', yaxis_title='Churn Rate (%)',width=700,height=500)
fig.show()


## Step 4. Identify high-churn segements (80/20 rule)

In [13]:
print("Segment with the highest churn rate by dimension:\n")

dimensions = [
    'acquisition_channel', 
    'product_tier', 
    'sales_segment',
    'region', 
    'company_size_bucket', 
    'onboarding_score_bucket'
]

for dim in dimensions:
    if dim not in df.columns:
        print(f"{dim}: Column not found in dataframe")
        continue

    # Compute churn rate per segment
    churn_rate = df.groupby(dim)['is_churned'].mean() * 100

    # Find the segment with highest churn
    top_segment = churn_rate.idxmax()
    top_rate = churn_rate.max()

    print(f"{dim}: {top_segment} ({top_rate:.1f}%)")

Segment with the highest churn rate by dimension:

acquisition_channel: Partner (12.7%)
product_tier: Starter (19.5%)
sales_segment: SMB_Field (16.9%)
region: North America (12.2%)
company_size_bucket: 1-10 (26.5%)
onboarding_score_bucket: Low (17.3%)
